<a href="https://colab.research.google.com/github/Ashleylq/plant-disease-classifier/blob/main/notebooks/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

In [ ]:
! pip install datasets torch torchvision matplotlib numpy scikit-learn

In [ ]:
from datasets import load_dataset
from collections import Counter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import v2 as transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.models as models
import numpy as np
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# **Load Dataset**

In [ ]:
dataset = load_dataset("BrandonFors/Plant-Diseases-PlantVillage-Dataset")
split = dataset["train"].train_test_split(test_size = 0.2, stratify_by_column="label")
train = split["train"]
val = split["test"]
test = dataset["test"]
dataset

In [ ]:
images = train[:5]["image"]
for image in images:
  display(image)

In [ ]:
train.features

# **Class Imbalance**

In [ ]:
class_counts = Counter(row["label"] for row in train)

labels, counts = zip(*class_counts.most_common())
labels = [train.features["label"].names[i] for i in labels]

labels

In [ ]:
plt.figure(figsize=(15, 8))
plt.bar(labels, counts)
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.show()

#**Prepare Data**

In [ ]:
def baseline_transform(batch):
  pipeline = transforms.Compose([
      transforms.ToImage(),
      transforms.ToDtype(torch.float32, scale=True)
  ])
  batch["image"] = [pipeline(img) for img in batch["image"]]
  return batch

train_ds = train.with_transform(baseline_transform)
train_ds[0]["image"]

In [ ]:
init_loader = DataLoader(train_ds, batch_size=64, num_workers=2)

def get_mean_std(Loader):
  running_sum = torch.zeros(3)
  running_sq_sum = torch.zeros(3)
  t_pixels = 0

  for batch in Loader:
    images = batch["image"]
    bs, c, h, w = images.shape
    p = bs * h * w
    images = images.view(c, -1)
    running_sum += images.sum(dim=1)
    running_sq_sum += (images ** 2).sum(dim=1)
    t_pixels += p

  mean = running_sum / t_pixels
  std = torch.sqrt((running_sq_sum / t_pixels) - (mean ** 2))
  return mean.tolist(), std.tolist()

mean, std = get_mean_std(init_loader)

In [ ]:
mean, std

# **Build a Simple CNN**

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=(224,224),antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
val_ds = val.with_transform(baseline_transform)

train_loader = DataLoader(train_ds, batch_size=64, num_workers=2, pin_memory=True, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=64, num_workers=2, pin_memory=True, shuffle=False)

In [ ]:
class Net(nn.Module):
  def __init__(self, num_classes=38):
    super(Net, self).__init__()
    self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
    self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
    self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
    self.pool = nn.MaxPool2d(2,2)
    self.linear1 = nn.Linear(128 * 28 * 28, 512)
    self.linear2 = nn.Linear(512, num_classes)
    self.dropout = nn.Dropout(p=0.5)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = self.pool(F.relu(self.conv3(x)))
    x = x.view(x.size(0), -1)
    x = F.relu(self.linear1(x))
    x = self.dropout(x)
    x = self.linear2(x)
    return x

model = Net()
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# **Training Loop**

In [ ]:
def train(model, epoch, TrainLoader, ValLoader, TrainTransforms, ValTransforms, optimizer, criterion, scheduler=False):
  train_loss = []
  val_loss = []
  for e in range(epoch):
    model.train()
    e_train_loss = 0
    for batch in TrainLoader:
      images = batch["image"].to(device)
      labels = batch["label"].to(device)
      images = TrainTransforms(images)
      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      e_train_loss += loss.item() * images.size(0)
      loss.backward()
      optimizer.step()
    e_train_loss = e_train_loss / len(TrainLoader.dataset)
    if scheduler:
      scheduler.step()

    model.eval()
    e_val_loss = 0
    with torch.no_grad():
      for batch in ValLoader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        images = ValTransforms(images)
        outputs = model(images)
        loss = criterion(outputs, labels)
        e_val_loss = loss.item() * images.size(0)
    e_val_loss = e_val_loss / len(ValLoader.dataset)

    print(f"epoch: {e}")
    print(f"train loss: {e_train_loss}")
    print(f"val loss: {e_val_loss}")
    train_loss.append(e_train_loss)
    val_loss.append(e_val_loss)

  epoch_range = range(epoch)
  plt.plot(epoch_range, train_loss, label="train")
  plt.plot(epoch_range, val_loss, label="val")
  plt.legend()
  plt.show()

In [ ]:
train(model, 20, train_loader, val_loader, train_transform, val_transform, optimizer, criterion)

# **Fine Tuning a pretrained model**

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=(224,224),antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
train_loader = DataLoader(train_ds, batch_size=256, num_workers=2, pin_memory=True, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=256, num_workers=2, pin_memory=True, shuffle=False)

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
for param in model.parameters():
  param.requires_grad = False
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 38)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

train(model, 20, train_loader, val_loader, train_transform, val_transform, optimizer, criterion, scheduler=scheduler)

# **Upload to Hugging Faces**

In [ ]:
!pip install huggingface_hub safetensors

In [ ]:
from huggingface_hub import HfApi
from safetensors.torch import save_file

state_dict = model.state_dict()
path = "model.safetensors"
save_file(state_dict, path)

api = HfApi()
api.upload_file(
    path_or_fileobj=path,
    path_in_repo="model.safetensors",
    repo_id="imightlikelemonade-97/plantvillage-resnet50",
    repo_type="model"
)

# **Evaluate Model**

In [ ]:
test_dataset = test.with_transform(baseline_transform)
test_loader = DataLoader(test_dataset, num_workers=2, batch_size=64, pin_memory=True, shuffle=False)

In [ ]:
model.eval()

true = []
pred = []
conf = []

with torch.no_grad():
  for batch in test_loader:
    images = batch["image"].to(device)
    labels = batch["label"].to(device)
    outputs = model(images)
    logits = outputs
    probs = F.softmax(logits, dim=-1)
    confidences, predictions = torch.max(probs, dim=-1)
    true.extend(labels.cpu().numpy())
    pred.extend(predictions.cpu().numpy())
    conf.extend(confidences.cpu().numpy())

true = np.array(true)
pred = np.array(pred)
conf = np.array(conf)

In [ ]:
class_names = test_dataset.features["label"].names
print(classification_report(true, pred, target_names=class_names))

In [ ]:
inc_idx = np.where((pred != true) & (conf > 0.9))[0]
print(len(inc_idx))

test_dataset.reset_format()
for idx in inc_idx[:5]:
  img = test_dataset[int(idx)]["image"]
  plt.figure(figsize=(5,5))
  plt.imshow(img)
  print(f"True: {true[int(idx)]}")
  print(f"Predicted: {pred[int(idx)]}")
  print(f"Confidence: {conf[int(idx)]}")
  plt.axis('off')
  plt.show()